https://business-science.github.io/pytimetk/tutorials/07_timeseries_crossvalidation.html

In [1]:
import pytimetk

In [2]:
# pip install pytimetk
# pip install --upgrade --force-reinstall statsmodels
# pip install --upgrade --force-reinstall pytimetk


# Intro

In [27]:
import pytimetk as tk
import pandas as pd

df = tk.load_dataset('bike_sales_sample')
df['order_date'] = pd.to_datetime(df['order_date'])

df.head(2)

,order_id,order_line,order_date,quantity,price,total_price,model,category_1,category_2,frame_material,bikeshop_name,city,state
0,1,1,2011-01-07,1,6070,6070,Jekyll Carbon 2,Mountain,Over Mountain,Carbon,Ithaca Mountain Climbers,Ithaca,NY
1,1,2,2011-01-07,1,5970,5970,Trigger Carbon 2,Mountain,Over Mountain,Carbon,Ithaca Mountain Climbers,Ithaca,NY


In [28]:
summary_category_1_df = df \
    .groupby("category_1") \
    .summarize_by_time(
        date_column  = 'order_date', 
        value_column = 'total_price',
        freq         = "MS",
        agg_func     = 'sum',
        wide_format  = False
    )

# First 5 rows shown
summary_category_1_df.head()

,category_1,order_date,total_price
0,Mountain,2011-01-01,221490
1,Mountain,2011-02-01,660555
2,Mountain,2011-03-01,358855
3,Mountain,2011-04-01,1075975
4,Mountain,2011-05-01,450440


In [29]:
summary_category_1_df \
    .groupby('category_1') \
    .plot_timeseries(
        date_column  = 'order_date',
        value_column = 'total_price',
        smooth_frac  = 0.8
    )

In [30]:
import pandas as pd
import numpy as np
import pytimetk as tk

from sklearn.ensemble import RandomForestRegressor

In [31]:
# We start by loading the dataset
# /walmart_sales_weekly.html
dset = tk.load_dataset('walmart_sales_weekly', parse_dates = ['Date'])

dset = dset.drop(columns=[
    'id', # This column can be removed as it is equivalent to 'Dept'
    'Store', # This column has only one possible value
    'Type', # This column has only one possible value
    'Size', # This column has only one possible value
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'IsHoliday', 'Temperature', 'Fuel_Price', 'CPI',
       'Unemployment'])

dset.head()

,Dept,Date,Weekly_Sales
0,1,2010-02-05,24924.50
1,1,2010-02-12,46039.49
2,1,2010-02-19,41595.55
3,1,2010-02-26,19403.54
4,1,2010-03-05,21827.90


In [32]:
sales_df = dset
fig = sales_df.groupby('Dept').plot_timeseries(
    date_column='Date',
    value_column='Weekly_Sales',
    facet_ncol = 2,
    x_axis_date_labels = "%Y",
    engine = 'plotly')
fig

In [33]:
print(sales_df.groupby('Dept').Date.max())

Dept
1    2012-10-26
3    2012-10-26
8    2012-10-26
13   2012-10-26
38   2012-10-26
93   2012-10-26
95   2012-10-26
Name: Date, dtype: datetime64[ns]


When building machine learning models, we need to setup our dataframe to hold information about the future. This is the dataframe that will get passed to our model.predict() call. This is made easy with tk.future_frame().

In [34]:
sales_df_with_futureframe = sales_df \
    .groupby('Dept') \
    .future_frame(
        date_column = 'Date',
        length_out  = 5
    )

Future framing...:   0%|          | 0/7 [00:00<?, ?it/s]

In [35]:
sales_df_with_futureframe.groupby('Dept').Date.max()


Dept
1    2012-11-30
3    2012-11-30
8    2012-11-30
13   2012-11-30
38   2012-11-30
93   2012-11-30
95   2012-11-30
Name: Date, dtype: datetime64[ns]

2.2 Date Features with tk.augment_timeseries_signature


In [36]:
sales_df_dates = sales_df_with_futureframe.augment_timeseries_signature(date_column = 'Date')
sales_df_dates.head(10)

,Dept,Date,Weekly_Sales,Date_index_num,Date_year,Date_year_iso,Date_yearstart,Date_yearend,Date_leapyear,Date_half,...,Date_mday,Date_qday,Date_yday,Date_weekend,Date_hour,Date_minute,Date_second,Date_msecond,Date_nsecond,Date_am_pm
0,1,2010-02-05,24924.50,1265328000,2010,2010,0,0,0,1,...,5,36,36,0,0,0,0,0,0,am
1,1,2010-02-12,46039.49,1265932800,2010,2010,0,0,0,1,...,12,43,43,0,0,0,0,0,0,am
2,1,2010-02-19,41595.55,1266537600,2010,2010,0,0,0,1,...,19,50,50,0,0,0,0,0,0,am
3,1,2010-02-26,19403.54,1267142400,2010,2010,0,0,0,1,...,26,57,57,0,0,0,0,0,0,am
4,1,2010-03-05,21827.90,1267747200,2010,2010,0,0,0,1,...,5,64,64,0,0,0,0,0,0,am
5,1,2010-03-12,21043.39,1268352000,2010,2010,0,0,0,1,...,12,71,71,0,0,0,0,0,0,am
6,1,2010-03-19,22136.64,1268956800,2010,2010,0,0,0,1,...,19,78,78,0,0,0,0,0,0,am
7,1,2010-03-26,26229.21,1269561600,2010,2010,0,0,0,1,...,26,85,85,0,0,0,0,0,0,am
8,1,2010-04-02,57258.43,1270166400,2010,2010,0,0,0,1,...,2,2,92,0,0,0,0,0,0,am
9,1,2010-04-09,42960.91,1270771200,2010,2010,0,0,0,1,...,9,9,99,0,0,0,0,0,0,am


In [37]:
sales_df_dates.glimpse()


<class 'pandas.core.frame.DataFrame'>: 1036 rows of 32 columns
Dept:               int64             [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  ...
Date:               datetime64[ns]    [Timestamp('2010-02-05 00:00:00'), ...
Weekly_Sales:       float64           [24924.5, 46039.49, 41595.55, 1940 ...
Date_index_num:     int64             [1265328000, 1265932800, 126653760 ...
Date_year:          int32             [2010, 2010, 2010, 2010, 2010, 201 ...
Date_year_iso:      UInt32            [2010, 2010, 2010, 2010, 2010, 201 ...
Date_yearstart:     uint8             [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  ...
Date_yearend:       uint8             [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  ...
Date_leapyear:      uint8             [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  ...
Date_half:          int64             [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  ...
Date_quarter:       int32             [1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2,  ...
Date_quarteryear:   object            ['2010Q1', '2010Q1', '2010Q1', '20 ...
Date_quarters

In [38]:
sales_df_dates = sales_df_dates[[
    'Date'
    ,'Dept'
    , 'Weekly_Sales'
    , 'Date_year'
    , 'Date_month'
    , 'Date_yweek'
    , 'Date_mweek'  
    ]]
sales_df_dates.tail(10)

,Date,Dept,Weekly_Sales,Date_year,Date_month,Date_yweek,Date_mweek
1026,2012-11-02,93,NaN,2012,11,44,1
1027,2012-11-09,93,NaN,2012,11,45,2
1028,2012-11-16,93,NaN,2012,11,46,3
1029,2012-11-23,93,NaN,2012,11,47,4
1030,2012-11-30,93,NaN,2012,11,48,5
1031,2012-11-02,95,NaN,2012,11,44,1
1032,2012-11-09,95,NaN,2012,11,45,2
1033,2012-11-16,95,NaN,2012,11,46,3
1034,2012-11-23,95,NaN,2012,11,47,4
1035,2012-11-30,95,NaN,2012,11,48,5


In [39]:
df_with_lags = sales_df_dates \
    .groupby('Dept') \
    .augment_lags(
        date_column  = 'Date',
        value_column = 'Weekly_Sales',
        lags         = [5,6,7,8,9]
    )
df_with_lags.head(5)

,Date,Dept,Weekly_Sales,Date_year,Date_month,Date_yweek,Date_mweek,Weekly_Sales_lag_5,Weekly_Sales_lag_6,Weekly_Sales_lag_7,Weekly_Sales_lag_8,Weekly_Sales_lag_9
0,2010-02-05,1,24924.50,2010,2,5,1,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,1,46039.49,2010,2,6,2,NaN,NaN,NaN,NaN,NaN
2,2010-02-19,1,41595.55,2010,2,7,3,NaN,NaN,NaN,NaN,NaN
3,2010-02-26,1,19403.54,2010,2,8,4,NaN,NaN,NaN,NaN,NaN
4,2010-03-05,1,21827.90,2010,3,9,1,NaN,NaN,NaN,NaN,NaN


In [40]:
lag_columns = [col for col in df_with_lags.columns if 'lag' in col]

df_with_rolling = df_with_lags \
    .groupby('Dept') \
    .augment_rolling(
        date_column  = 'Date',
        value_column = lag_columns,
        window  = 4,
        window_func = 'mean',
        threads = 1 # Change to -1 to use all available cores
    ) 
df_with_rolling[df_with_rolling.Dept ==1].head(10)

Calculating Rolling...:   0%|          | 0/7 [00:00<?, ?it/s]

,Date,Dept,Weekly_Sales,Date_year,Date_month,Date_yweek,Date_mweek,Weekly_Sales_lag_5,Weekly_Sales_lag_6,Weekly_Sales_lag_7,Weekly_Sales_lag_8,Weekly_Sales_lag_9,Weekly_Sales_lag_5_rolling_mean_win_4,Weekly_Sales_lag_6_rolling_mean_win_4,Weekly_Sales_lag_7_rolling_mean_win_4,Weekly_Sales_lag_8_rolling_mean_win_4,Weekly_Sales_lag_9_rolling_mean_win_4
0,2010-02-05,1,24924.50,2010,2,5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,2010-02-05,1,24924.50,2010,2,5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,2010-02-05,1,24924.50,2010,2,5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,2010-02-05,1,24924.50,2010,2,5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,2010-02-05,1,24924.50,2010,2,5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,1,46039.49,2010,2,6,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,1,46039.49,2010,2,6,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,1,46039.49,2010,2,6,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,1,46039.49,2010,2,6,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,1,46039.49,2010,2,6,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
all_lag_columns = [col for col in df_with_rolling.columns if 'lag' in col]

df_no_nas = df_with_rolling \
    .dropna(subset=all_lag_columns, inplace=False)

df_no_nas.head()

,Date,Dept,Weekly_Sales,Date_year,Date_month,Date_yweek,Date_mweek,Weekly_Sales_lag_5,Weekly_Sales_lag_6,Weekly_Sales_lag_7,Weekly_Sales_lag_8,Weekly_Sales_lag_9,Weekly_Sales_lag_5_rolling_mean_win_4,Weekly_Sales_lag_6_rolling_mean_win_4,Weekly_Sales_lag_7_rolling_mean_win_4,Weekly_Sales_lag_8_rolling_mean_win_4,Weekly_Sales_lag_9_rolling_mean_win_4
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.9,19403.54,22809.285,21102.8675,25967.595,32216.62,32990.77
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.9,19403.54,22809.285,21102.8675,25967.595,32216.62,32990.77
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.9,19403.54,22809.285,21102.8675,25967.595,32216.62,32990.77
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.9,19403.54,22809.285,21102.8675,25967.595,32216.62,32990.77
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.9,19403.54,22809.285,21102.8675,25967.595,32216.62,32990.77


In [42]:
df_no_nas.glimpse()

<class 'pandas.core.frame.DataFrame'>: 4760 rows of 17 columns
Date:                                   datetime64[ns]    [Timestamp('20 ...
Dept:                                   int64             [1, 1, 1, 1, 1 ...
Weekly_Sales:                           float64           [16555.11, 165 ...
Date_year:                              int32             [2010, 2010, 2 ...
Date_month:                             int32             [4, 4, 4, 4, 4 ...
Date_yweek:                             UInt32            [17, 17, 17, 1 ...
Date_mweek:                             int32             [5, 5, 5, 5, 5 ...
Weekly_Sales_lag_5:                     float64           [26229.21, 262 ...
Weekly_Sales_lag_6:                     float64           [22136.64, 221 ...
Weekly_Sales_lag_7:                     float64           [21043.39, 210 ...
Weekly_Sales_lag_8:                     float64           [21827.9, 2182 ...
Weekly_Sales_lag_9:                     float64           [19403.54, 194 ...
Weekly_Sales_

In [43]:
future = df_no_nas[df_no_nas.Weekly_Sales.isnull()]
train = df_no_nas[df_no_nas.Weekly_Sales.notnull()]

In [44]:
train_columns = [ 
    'Dept'
    , 'Date_year'
    , 'Date_month'
    , 'Date_yweek'
    , 'Date_mweek'
    , 'Weekly_Sales_lag_5'
    , 'Weekly_Sales_lag_6'
    , 'Weekly_Sales_lag_7'
    , 'Weekly_Sales_lag_8'
    , 'Weekly_Sales_lag_5_rolling_mean_win_4'
    , 'Weekly_Sales_lag_6_rolling_mean_win_4'
    , 'Weekly_Sales_lag_7_rolling_mean_win_4'
    , 'Weekly_Sales_lag_8_rolling_mean_win_4'
    ]

X = train[train_columns]
y = train[['Weekly_Sales']]

model = RandomForestRegressor(random_state=123)
model = model.fit(X, y)

In [45]:
predicted_values = model.predict(future[train_columns])
future['y_pred'] = predicted_values

future.head(10)

,Date,Dept,Weekly_Sales,Date_year,Date_month,Date_yweek,Date_mweek,Weekly_Sales_lag_5,Weekly_Sales_lag_6,Weekly_Sales_lag_7,Weekly_Sales_lag_8,Weekly_Sales_lag_9,Weekly_Sales_lag_5_rolling_mean_win_4,Weekly_Sales_lag_6_rolling_mean_win_4,Weekly_Sales_lag_7_rolling_mean_win_4,Weekly_Sales_lag_8_rolling_mean_win_4,Weekly_Sales_lag_9_rolling_mean_win_4,y_pred
1001,2012-11-02,1,NaN,2012,11,44,1,18947.81,19251.50,19616.22,18322.37,16680.24,19034.475,18467.5825,17726.3075,17154.9275,16604.3150,26627.7378
1001,2012-11-02,1,NaN,2012,11,44,1,18947.81,19251.50,19616.22,18322.37,16680.24,19034.475,18467.5825,17726.3075,17154.9275,16604.3150,26627.7378
1001,2012-11-02,1,NaN,2012,11,44,1,18947.81,19251.50,19616.22,18322.37,16680.24,19034.475,18467.5825,17726.3075,17154.9275,16604.3150,26627.7378
1001,2012-11-02,1,NaN,2012,11,44,1,18947.81,19251.50,19616.22,18322.37,16680.24,19034.475,18467.5825,17726.3075,17154.9275,16604.3150,26627.7378
1001,2012-11-02,1,NaN,2012,11,44,1,18947.81,19251.50,19616.22,18322.37,16680.24,19034.475,18467.5825,17726.3075,17154.9275,16604.3150,26627.7378
1002,2012-11-09,1,NaN,2012,11,45,2,21904.47,18947.81,19251.50,19616.22,18322.37,19930.000,19034.4750,18467.5825,17726.3075,17154.9275,20959.0553
1002,2012-11-09,1,NaN,2012,11,45,2,21904.47,18947.81,19251.50,19616.22,18322.37,19930.000,19034.4750,18467.5825,17726.3075,17154.9275,20959.0553
1002,2012-11-09,1,NaN,2012,11,45,2,21904.47,18947.81,19251.50,19616.22,18322.37,19930.000,19034.4750,18467.5825,17726.3075,17154.9275,20959.0553
1002,2012-11-09,1,NaN,2012,11,45,2,21904.47,18947.81,19251.50,19616.22,18322.37,19930.000,19034.4750,18467.5825,17726.3075,17154.9275,20959.0553
1002,2012-11-09,1,NaN,2012,11,45,2,21904.47,18947.81,19251.50,19616.22,18322.37,19930.000,19034.4750,18467.5825,17726.3075,17154.9275,20959.0553


In [46]:
train['type'] = 'actuals'
future['type'] = 'prediction'

full_df = pd.concat([train, future])

full_df.head(10)

,Date,Dept,Weekly_Sales,Date_year,Date_month,Date_yweek,Date_mweek,Weekly_Sales_lag_5,Weekly_Sales_lag_6,Weekly_Sales_lag_7,Weekly_Sales_lag_8,Weekly_Sales_lag_9,Weekly_Sales_lag_5_rolling_mean_win_4,Weekly_Sales_lag_6_rolling_mean_win_4,Weekly_Sales_lag_7_rolling_mean_win_4,Weekly_Sales_lag_8_rolling_mean_win_4,Weekly_Sales_lag_9_rolling_mean_win_4,type,y_pred
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.90,19403.54,22809.2850,21102.8675,25967.5950,32216.620,32990.77,actuals,NaN
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.90,19403.54,22809.2850,21102.8675,25967.5950,32216.620,32990.77,actuals,NaN
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.90,19403.54,22809.2850,21102.8675,25967.5950,32216.620,32990.77,actuals,NaN
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.90,19403.54,22809.2850,21102.8675,25967.5950,32216.620,32990.77,actuals,NaN
12,2010-04-30,1,16555.11,2010,4,17,5,26229.21,22136.64,21043.39,21827.90,19403.54,22809.2850,21102.8675,25967.5950,32216.620,32990.77,actuals,NaN
13,2010-05-07,1,17413.94,2010,5,18,1,57258.43,26229.21,22136.64,21043.39,21827.90,31666.9175,22809.2850,21102.8675,25967.595,32216.62,actuals,NaN
13,2010-05-07,1,17413.94,2010,5,18,1,57258.43,26229.21,22136.64,21043.39,21827.90,31666.9175,22809.2850,21102.8675,25967.595,32216.62,actuals,NaN
13,2010-05-07,1,17413.94,2010,5,18,1,57258.43,26229.21,22136.64,21043.39,21827.90,31666.9175,22809.2850,21102.8675,25967.595,32216.62,actuals,NaN
13,2010-05-07,1,17413.94,2010,5,18,1,57258.43,26229.21,22136.64,21043.39,21827.90,31666.9175,22809.2850,21102.8675,25967.595,32216.62,actuals,NaN
13,2010-05-07,1,17413.94,2010,5,18,1,57258.43,26229.21,22136.64,21043.39,21827.90,31666.9175,22809.2850,21102.8675,25967.595,32216.62,actuals,NaN


In [47]:
full_df['Weekly_Sales'] = np.where(full_df.type =='actuals', full_df.Weekly_Sales, full_df.y_pred)

In [48]:
full_df \
    .groupby('Dept') \
    .plot_timeseries(
        date_column = 'Date',
        value_column = 'Weekly_Sales',
        color_column = 'type',
        smooth = False,
        smooth_alpha = 0,
        facet_ncol = 2,
        facet_scales = "free",
        y_intercept_color = tk.palette_timetk()['steel_blue'],
        width = 800,
        height = 600,
        engine = 'plotly'
    )

Here are some additional techniques that can be explored to elevate its performance:

Experiment with the incorporation of various lags using the versatile tk.augment_lags() function.

Enhance the model’s capabilities by introducing additional rolling calculations through tk.augment_rolling().

Consider incorporating cyclic features by utilizing tk.augment_fourier().

Try different models and build a robust cross-validation strategy for model selection.

These strategies hold promise for refining the model’s accuracy and predictive power